# 03 - Non-Recurrent Control

Builds the matched zero-recurrence control used to isolate the contribution of recurrence in RQ1. See Methodology Section 3.2 and Results Section 4.2.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import time

os.environ["RESERVOIRPY_VERBOSITY"] = "0"

import joblib
import numpy as np
import reservoirpy as rpy
import scipy.sparse as sp

from reservoirpy.nodes import Reservoir
from scipy.stats import binomtest, chi2
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.preprocessing import StandardScaler

SEED = 42
rpy.set_seed(SEED)
np.random.seed(SEED)

print(f"ReservoirPy: {rpy.__version__}")

ReservoirPy: 0.4.1


## 1. Configuration

Loads the locked reservoir configuration and the completed Notebook 02 run without retuning anything.

In [2]:
def find_project_root() -> Path:
    """Find the repository root from the data/nsynth folder."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "nsynth").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/nsynth. Start Jupyter inside the repository."
    )


PROJECT_ROOT = find_project_root()
CACHE_ROOT = PROJECT_ROOT / "cache_final"
SELECTION_MANIFEST_PATH = (
    PROJECT_ROOT / "results_sweep_final" / "selected_reservoir_config.json"
)

LATEST_RUN_PATH = PROJECT_ROOT / "results_final" / "latest_run.json"
if not LATEST_RUN_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 before Notebook 03.")
with LATEST_RUN_PATH.open("r", encoding="utf-8") as f:
    RUN_TAG = json.load(f)["run_tag"]
EXISTING_RUN_DIR = PROJECT_ROOT / "results_final" / f"run_{RUN_TAG}"
OUTPUT_DIR = PROJECT_ROOT / "results_nonrecurrent_control"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAMILY_NAMES = [
    "bass", "brass", "flute", "guitar", "keyboard",
    "mallet", "organ", "reed", "string", "vocal",
]

if not SELECTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing locked reservoir manifest: {SELECTION_MANIFEST_PATH}"
    )
with open(SELECTION_MANIFEST_PATH) as f:
    SELECTION_MANIFEST = json.load(f)

if SELECTION_MANIFEST.get("selection_status") != "locked":
    raise RuntimeError("Reservoir-selection manifest is not locked.")
if int(SELECTION_MANIFEST.get("reservoir_seed")) != SEED:
    raise RuntimeError("Reservoir seed differs from the locked selection manifest.")

FEATURE_CFG = dict(SELECTION_MANIFEST["feature_cfg"])
SR = FEATURE_CFG["sr"]
N_MELS = FEATURE_CFG["n_mels"]
MAX_T = FEATURE_CFG["max_t"]
USE_DELTAS = FEATURE_CFG["use_deltas"]
FEAT_DIM = N_MELS * (3 if USE_DELTAS else 1)

RESERVOIR_CFG = dict(SELECTION_MANIFEST["reservoir_cfg"])
RESERVOIR_UNITS = int(RESERVOIR_CFG["units"])

RIDGE_ALPHAS = list(SELECTION_MANIFEST["search_design"]["ridge_alphas"])
RIDGE_CLASS_WEIGHT = "balanced"
RIDGE_SOLVER = SELECTION_MANIFEST["search_design"]["ridge_solver"]
RIDGE_TOL = float(SELECTION_MANIFEST["search_design"]["ridge_tol"])
RIDGE_MAX_ITER = int(SELECTION_MANIFEST["search_design"]["ridge_max_iter"])
RIDGE_TIE_ATOL = 1e-12
VALIDATION_METRIC = "macro_f1"
SCALER_N_EXAMPLES = 3_000


def stable_hash(obj) -> str:
    txt = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha1(txt.encode("utf-8")).hexdigest()[:10]


FEATURE_TAG = stable_hash(FEATURE_CFG)
if FEATURE_TAG != SELECTION_MANIFEST["feature_tag"]:
    raise RuntimeError("Feature tag does not match notebook 01.")

SCALER_CFG = {
    "feature_tag": FEATURE_TAG,
    "scaler_n_examples": SCALER_N_EXAMPLES,
    "seed": SEED,
}
SCALER_TAG = stable_hash(SCALER_CFG)
if SCALER_TAG != SELECTION_MANIFEST["scaler_tag"]:
    raise RuntimeError("Feature-scaler tag does not match notebook 01.")

MEL_CACHE_DIR = CACHE_ROOT / f"mel_{FEATURE_TAG}"
FEATURE_SCALER_PATH = MEL_CACHE_DIR / f"feature_scaler_{SCALER_TAG}.joblib"

if not MEL_CACHE_DIR.exists() or not FEATURE_SCALER_PATH.exists():
    raise FileNotFoundError(
        "Expected existing acoustic feature cache/scaler not found -- "
        "run notebooks 01/02 first. Refusing to rebuild the shared "
        f"feature cache: {MEL_CACHE_DIR}"
    )

# New, non-colliding configuration hash for this control arm.
CONTROL_CFG = {
    "arm": "nonrecurrent_control",
    "base_reservoir_cfg": RESERVOIR_CFG,
    "recurrent_weight_matrix": "zeroed",
    "leak_rate": 1.0,
    "feature_tag": FEATURE_TAG,
    "scaler_tag": SCALER_TAG,
    "seed": SEED,
    "scaler_n_examples": SCALER_N_EXAMPLES,
}
CONTROL_TAG = stable_hash(CONTROL_CFG)
CONTROL_CACHE_DIR = CACHE_ROOT / f"states_nonrecurrent_control_{CONTROL_TAG}"
CONTROL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Locked reservoir cfg:   {RESERVOIR_CFG}")
print(f"Feature cache (reused): {MEL_CACHE_DIR}")
print(f"Control cache (new):    {CONTROL_CACHE_DIR}")
print(f"Control tag:            {CONTROL_TAG}")
print(f"Output dir:             {OUTPUT_DIR}")

Locked reservoir cfg:   {'units': 1000, 'sr': 0.99, 'lr': 0.05, 'input_scaling': 0.2, 'rc_connectivity': 0.1}
Feature cache (reused): /home/olliechandler/ESN-NSYNTH/cache_final/mel_8cfbd6d555
Control cache (new):    /home/olliechandler/ESN-NSYNTH/cache_final/states_nonrecurrent_control_5d3b246266
Control tag:            5d3b246266
Output dir:             /home/olliechandler/ESN-NSYNTH/results_nonrecurrent_control


## 2. Cached acoustic inputs

Reuses the acoustic caches and labels created by Notebook 02.

In [3]:
train_mel_X = MEL_CACHE_DIR / "X_train.npy"
valid_mel_X = MEL_CACHE_DIR / "X_valid.npy"
test_mel_X = MEL_CACHE_DIR / "X_test.npy"
train_y_path = MEL_CACHE_DIR / "y_train.npy"
valid_y_path = MEL_CACHE_DIR / "y_valid.npy"
test_y_path = MEL_CACHE_DIR / "y_test.npy"

for p in (train_mel_X, valid_mel_X, test_mel_X, train_y_path, valid_y_path, test_y_path):
    if not p.exists():
        raise FileNotFoundError(f"Missing expected cached file: {p}")


def load_labels(y_path: Path) -> np.ndarray:
    return np.asarray(np.load(y_path, mmap_mode="r"), dtype=np.int64)


y_train = load_labels(train_y_path)
y_valid = load_labels(valid_y_path)
y_test = load_labels(test_y_path)

print(f"Train: {len(y_train):,}  Valid: {len(y_valid):,}  Test: {len(y_test):,}")
assert (len(y_train), len(y_valid), len(y_test)) == (283_704, 12_678, 4_096), (
    "Cached partitions do not match the required 283,704 / 12,678 / 4,096 split."
)

feature_scaler = joblib.load(FEATURE_SCALER_PATH)

Train: 283,704  Valid: 12,678  Test: 4,096


## 3. Matched non-recurrent system

Builds the same seeded input projection, then sets the recurrent matrix to zero and the leak rate to one.

In [4]:
def build_reservoir():
    reservoir = Reservoir(**RESERVOIR_CFG, seed=SEED)
    reservoir.run(np.zeros((3, FEAT_DIM), dtype=np.float32))
    return reservoir


reference_reservoir = build_reservoir()
control_reservoir = build_reservoir()

win_ref = (
    reference_reservoir.Win.toarray()
    if sp.issparse(reference_reservoir.Win)
    else np.asarray(reference_reservoir.Win)
)
win_ctrl = (
    control_reservoir.Win.toarray()
    if sp.issparse(control_reservoir.Win)
    else np.asarray(control_reservoir.Win)
)
win_matches = bool(np.array_equal(win_ref, win_ctrl))
if not win_matches:
    raise RuntimeError(
        "Control reservoir's W_in does not match the reference ESN reservoir's "
        "W_in produced by the same seeded ReservoirPy call. Stopping: the input "
        "projection is not matched as required."
    )
print(f"W_in matches reference ESN reservoir (same seeded call): {win_matches}")

control_reservoir.W = sp.csr_array(control_reservoir.W.shape, dtype=control_reservoir.W.dtype)
control_reservoir.lr = 1.0
w_is_zero = bool(control_reservoir.W.nnz == 0)
print(f"Control reservoir W zeroed (nnz=0): {w_is_zero}, lr={control_reservoir.lr}")

del reference_reservoir
gc.collect()


def run_control_on_feature(feature_sequence: np.ndarray) -> np.ndarray:
    feature_scaled = feature_scaler.transform(feature_sequence.astype(np.float32))
    control_reservoir.reset()
    states = control_reservoir.run(feature_scaled)
    return np.asarray(states, dtype=np.float32)

W_in matches reference ESN reservoir (same seeded call): True
Control reservoir W zeroed (nnz=0): True, lr=1.0


## 4. Memorylessness check

Verifies that shuffling time only permutes the control states, confirming that recurrence has been removed.

In [5]:
X_train_mmap = np.load(train_mel_X, mmap_mode="r")
assert_rng = np.random.default_rng(SEED)
check_idx = assert_rng.choice(len(X_train_mmap), size=25, replace=False)

max_abs_diff = 0.0
for i in check_idx:
    seq = np.asarray(X_train_mmap[i], dtype=np.float32)
    states_unshuffled = run_control_on_feature(seq)

    perm = assert_rng.permutation(len(seq))
    seq_shuffled = seq[perm]
    states_shuffled = run_control_on_feature(seq_shuffled)

    diff = np.max(np.abs(states_shuffled - states_unshuffled[perm]))
    max_abs_diff = max(max_abs_diff, float(diff))

memoryless_assertion_passed = bool(max_abs_diff < 1e-5)
print(
    f"Memorylessness assertion over {len(check_idx)} examples: "
    f"max |Δ| = {max_abs_diff:.3e}, passed = {memoryless_assertion_passed}"
)
if not memoryless_assertion_passed:
    raise RuntimeError(
        "Memorylessness assertion FAILED: shuffled-time states do not match "
        "unshuffled states under row permutation. The recurrent path has not "
        "been fully removed. Stopping."
    )
del X_train_mmap
gc.collect()

Memorylessness assertion over 25 examples: max |Δ| = 0.000e+00, passed = True


0

## 5. Control state scaler

Fits the same training-only 3,000-recording state-scaling protocol used for the ESN.

In [6]:
CONTROL_STATE_SCALER_PATH = CONTROL_CACHE_DIR / f"state_scaler_{CONTROL_TAG}.joblib"


def fit_or_load_control_state_scaler():
    if CONTROL_STATE_SCALER_PATH.exists():
        print(f"Loading control state scaler: {CONTROL_STATE_SCALER_PATH}")
        return joblib.load(CONTROL_STATE_SCALER_PATH)

    print("Fitting control state scaler from training reservoir states...")
    X = np.load(train_mel_X, mmap_mode="r")
    rng = np.random.default_rng(SEED)
    n = min(SCALER_N_EXAMPLES, len(X))
    indices = rng.choice(len(X), size=n, replace=False)

    scaler = StandardScaler()
    started = time.time()
    for j, i in enumerate(indices):
        states = run_control_on_feature(np.asarray(X[i], dtype=np.float32))
        scaler.partial_fit(states)
        if (j + 1) % 500 == 0 or j + 1 == len(indices):
            print(f"  scaler {j+1:,}/{len(indices):,} | {time.time()-started:.1f}s")

    joblib.dump(scaler, CONTROL_STATE_SCALER_PATH)
    del X
    gc.collect()
    print(f"Saved control state scaler: {CONTROL_STATE_SCALER_PATH}")
    return scaler


control_state_scaler = fit_or_load_control_state_scaler()

Fitting control state scaler from training reservoir states...
  scaler 500/3,000 | 1.5s
  scaler 1,000/3,000 | 3.0s
  scaler 1,500/3,000 | 4.5s
  scaler 2,000/3,000 | 6.0s
  scaler 2,500/3,000 | 7.5s
  scaler 3,000/3,000 | 9.0s
Saved control state scaler: /home/olliechandler/ESN-NSYNTH/cache_final/states_nonrecurrent_control_5d3b246266/state_scaler_5d3b246266.joblib


## 6. Pool the control states

Streams the control states and stores the mean, maximum and final-state summary used by the RQ1 Ridge classifier.

In [7]:
POOLED_DTYPE = np.float32


def pooled_cache_path(split_name: str) -> Path:
    return CONTROL_CACHE_DIR / f"X_pooled_{split_name}.npy"


def build_pooled_split(split_name: str, mel_path: Path, n_examples: int) -> np.ndarray:
    out_path = pooled_cache_path(split_name)
    expected_shape = (n_examples, RESERVOIR_UNITS * 3)
    if out_path.exists():
        arr = np.load(out_path, mmap_mode="r")
        if arr.shape == expected_shape and arr.dtype == np.dtype(POOLED_DTYPE):
            print(f"Using cached pooled control states for {split_name}: {out_path}")
            return np.asarray(arr, dtype=np.float32)

    print(f"Building pooled control states for {split_name}...")
    X_mel = np.load(mel_path, mmap_mode="r")
    out = np.lib.format.open_memmap(
        out_path, mode="w+", dtype=POOLED_DTYPE, shape=expected_shape
    )
    started = time.time()
    for i in range(n_examples):
        seq = np.asarray(X_mel[i], dtype=np.float32)
        states = run_control_on_feature(seq)
        states = control_state_scaler.transform(states)
        out[i] = np.concatenate(
            [states.mean(axis=0), states.max(axis=0), states[-1]]
        ).astype(POOLED_DTYPE)

        if (i + 1) % 20_000 == 0 or i + 1 == n_examples:
            elapsed = time.time() - started
            rate = (i + 1) / max(elapsed, 1e-9)
            print(f"  {split_name:5s} {i+1:,}/{n_examples:,} | {rate:.1f} examples/s")
            out.flush()

    out.flush()
    del X_mel
    result = np.asarray(out, dtype=np.float32)
    del out
    gc.collect()
    return result


Xtr_ctrl = build_pooled_split("train", train_mel_X, len(y_train))
Xva_ctrl = build_pooled_split("valid", valid_mel_X, len(y_valid))
Xte_ctrl = build_pooled_split("test", test_mel_X, len(y_test))

with open(CONTROL_CACHE_DIR / f"manifest_{CONTROL_TAG}.json", "w") as f:
    json.dump(
        {
            "control_tag": CONTROL_TAG,
            "control_cfg": CONTROL_CFG,
            "win_matches_reference": win_matches,
            "w_is_zero": w_is_zero,
            "memoryless_assertion": {
                "n_examples_checked": int(len(check_idx)),
                "max_abs_diff": max_abs_diff,
                "passed": memoryless_assertion_passed,
            },
            "n_train": int(len(y_train)),
            "n_valid": int(len(y_valid)),
            "n_test": int(len(y_test)),
        },
        f,
        indent=2,
    )

Building pooled control states for train...
  train 20,000/283,704 | 424.8 examples/s
  train 40,000/283,704 | 424.7 examples/s
  train 60,000/283,704 | 425.4 examples/s
  train 80,000/283,704 | 426.0 examples/s
  train 100,000/283,704 | 426.4 examples/s
  train 120,000/283,704 | 426.8 examples/s
  train 140,000/283,704 | 427.0 examples/s
  train 160,000/283,704 | 427.4 examples/s
  train 180,000/283,704 | 427.8 examples/s
  train 200,000/283,704 | 428.4 examples/s
  train 220,000/283,704 | 429.0 examples/s
  train 240,000/283,704 | 429.2 examples/s
  train 260,000/283,704 | 428.9 examples/s
  train 280,000/283,704 | 429.2 examples/s
  train 283,704/283,704 | 429.2 examples/s
Building pooled control states for valid...
  valid 12,678/12,678 | 433.4 examples/s
Building pooled control states for test...
  test  4,096/4,096 | 432.5 examples/s


## 7. Ridge readout

Uses the same Ridge grid, solver, class weighting and validation-selection rule as the main RQ1 experiments.

In [8]:
def classification_metrics(labels: np.ndarray, predictions: np.ndarray) -> dict:
    return {
        "acc": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(labels, predictions)),
    }


def select_ridge_candidate(rows: list) -> dict:
    best_score = max(row[VALIDATION_METRIC] for row in rows)
    tied = [
        row for row in rows
        if np.isclose(row[VALIDATION_METRIC], best_score, rtol=0.0, atol=RIDGE_TIE_ATOL)
    ]
    # Prefer stronger regularisation if predictions tie.
    return max(tied, key=lambda row: row["alpha"])


print("Fitting non-recurrent control Ridge readout...")
summary_scaler = StandardScaler().fit(Xtr_ctrl)
Xtr_s = summary_scaler.transform(Xtr_ctrl)
Xva_s = summary_scaler.transform(Xva_ctrl)
Xte_s = summary_scaler.transform(Xte_ctrl)

tuning_rows = []
models = {}
for alpha in RIDGE_ALPHAS:
    classifier = RidgeClassifier(
        alpha=alpha,
        class_weight=RIDGE_CLASS_WEIGHT,
        solver=RIDGE_SOLVER,
        tol=RIDGE_TOL,
        max_iter=RIDGE_MAX_ITER,
    )
    classifier.fit(Xtr_s, y_train)
    validation_prediction = classifier.predict(Xva_s)
    validation_metrics = classification_metrics(y_valid, validation_prediction)
    tuning_rows.append(
        {
            "alpha": float(alpha),
            **validation_metrics,
            "n_iter": (
                None if classifier.n_iter_ is None
                else np.asarray(classifier.n_iter_).tolist()
            ),
        }
    )
    models[float(alpha)] = classifier
    print(f"  alpha={alpha:.1e}  valid macro_f1={validation_metrics['macro_f1']*100:.2f}%")

selected = select_ridge_candidate(tuning_rows)
selected_alpha = float(selected["alpha"])
selected_model = models[selected_alpha]

# Test set touched exactly once, after alpha selection is complete.
control_test_predictions = selected_model.predict(Xte_s)
control_test_metrics = classification_metrics(y_test, control_test_predictions)

print(f"Selected alpha: {selected_alpha:.1e}")
print(f"Validation macro F1: {selected['macro_f1']*100:.2f}%")
print(f"Test accuracy:       {control_test_metrics['acc']*100:.2f}%")
print(f"Test macro F1:       {control_test_metrics['macro_f1']*100:.2f}%")
print(f"Test balanced acc:   {control_test_metrics['balanced_acc']*100:.2f}%")

per_class_report = classification_report(
    y_test, control_test_predictions,
    labels=list(range(len(FAMILY_NAMES))),
    target_names=FAMILY_NAMES, output_dict=True, zero_division=0,
)

Fitting non-recurrent control Ridge readout...
  alpha=1.0e-10  valid macro_f1=62.01%
  alpha=1.0e-09  valid macro_f1=62.01%
  alpha=1.0e-08  valid macro_f1=62.01%
  alpha=1.0e-07  valid macro_f1=62.01%
  alpha=1.0e-06  valid macro_f1=62.01%
  alpha=1.0e-05  valid macro_f1=62.01%
  alpha=1.0e-04  valid macro_f1=62.01%
  alpha=1.0e-03  valid macro_f1=62.01%
  alpha=1.0e-02  valid macro_f1=62.01%
  alpha=1.0e-01  valid macro_f1=62.01%
  alpha=1.0e+00  valid macro_f1=62.01%
  alpha=1.0e+01  valid macro_f1=62.00%
  alpha=1.0e+02  valid macro_f1=61.90%
  alpha=3.0e+02  valid macro_f1=61.87%
  alpha=1.0e+03  valid macro_f1=61.50%
  alpha=3.0e+03  valid macro_f1=61.28%
  alpha=1.0e+04  valid macro_f1=60.24%
Selected alpha: 1.0e+00
Validation macro F1: 62.01%
Test accuracy:       64.79%
Test macro F1:       62.13%
Test balanced acc:   65.86%


## 8. Save control results

Writes the control metrics, predictions and configuration used by the reporting notebooks.

In [9]:
results = {
    "config": {
        "control_tag": CONTROL_TAG,
        "base_reservoir_cfg": RESERVOIR_CFG,
        "recurrent_weight_matrix": "zeroed",
        "leak_rate": 1.0,
        "win_matches_reference_reservoir": win_matches,
        "w_is_zero": w_is_zero,
        "memoryless_assertion": {
            "n_examples_checked": int(len(check_idx)),
            "max_abs_diff": max_abs_diff,
            "passed": memoryless_assertion_passed,
        },
        "seed": SEED,
        "feature_tag": FEATURE_TAG,
        "scaler_tag": SCALER_TAG,
        "n_train": int(len(y_train)),
        "n_valid": int(len(y_valid)),
        "n_test": int(len(y_test)),
        "ridge_alphas": RIDGE_ALPHAS,
        "ridge_class_weight": RIDGE_CLASS_WEIGHT,
        "ridge_solver": RIDGE_SOLVER,
        "ridge_tol": RIDGE_TOL,
        "ridge_max_iter": RIDGE_MAX_ITER,
        "validation_metric": VALIDATION_METRIC,
    },
    "alpha": selected_alpha,
    "validation": {
        key: float(selected[key]) for key in ("acc", "macro_f1", "balanced_acc")
    },
    "test": control_test_metrics,
    "tuning_curve": tuning_rows,
    "test_predictions": control_test_predictions.astype(int).tolist(),
    "per_class": per_class_report,
}

results_path = OUTPUT_DIR / "results_nonrecurrent_control.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
np.save(OUTPUT_DIR / "test_predictions_nonrecurrent_control.npy", control_test_predictions)
print(f"Wrote {results_path}")

Wrote /home/olliechandler/ESN-NSYNTH/results_nonrecurrent_control/results_nonrecurrent_control.json


## 9. Paired McNemar tests

Compares the control with the acoustic and ESN-state Ridge predictions from Notebook 02.

In [10]:
existing_labels = np.load(EXISTING_RUN_DIR / "test_labels.npy")
existing_acoustic_preds = np.load(EXISTING_RUN_DIR / "test_predictions_acoustic_ridge.npy")
existing_esn_preds = np.load(EXISTING_RUN_DIR / "test_predictions_esn_ridge.npy")

if not np.array_equal(existing_labels, y_test):
    raise RuntimeError(
        "Existing test_labels.npy does not match this run's y_test -- the test "
        "partitions are not aligned example-for-example. Stopping rather than "
        "running McNemar tests on mismatched pairs."
    )
print("Existing test labels match this run's test labels (same order).")


def mcnemar_comparison(predictions_a, predictions_b, labels, name_a: str, name_b: str) -> dict:
    correct_a = np.asarray(predictions_a) == labels
    correct_b = np.asarray(predictions_b) == labels

    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    discordant = b + c

    if discordant == 0:
        corrected_statistic = 0.0
        corrected_p = 1.0
        exact_p = 1.0
    else:
        corrected_statistic = ((abs(b - c) - 1) ** 2) / discordant
        corrected_p = float(chi2.sf(corrected_statistic, df=1))
        exact_p = float(
            binomtest(min(b, c), n=discordant, p=0.5, alternative="two-sided").pvalue
        )

    print(f"{name_a} only correct: {b}")
    print(f"{name_b} only correct: {c}")
    print(f"Continuity-corrected chi-square: {corrected_statistic:.4f}")
    print(f"Corrected p-value:             {corrected_p:.3e}")
    print(f"Exact two-sided p-value:       {exact_p:.3e}")

    return {
        "b": b, "c": c, "discordant": discordant,
        "chi_square_corrected": float(corrected_statistic),
        "p_corrected": corrected_p,
        "p_exact": exact_p,
    }


print("Non-recurrent control vs acoustic-feature Ridge")
mcnemar_vs_acoustic = mcnemar_comparison(
    control_test_predictions, existing_acoustic_preds, y_test,
    "Non-recurrent control", "Acoustic Ridge",
)

print()
print("Non-recurrent control vs ESN-state Ridge")
mcnemar_vs_esn = mcnemar_comparison(
    control_test_predictions, existing_esn_preds, y_test,
    "Non-recurrent control", "ESN-state Ridge",
)

with open(OUTPUT_DIR / "mcnemar_nonrecurrent_control.json", "w") as f:
    json.dump(
        {
            "vs_acoustic_ridge": mcnemar_vs_acoustic,
            "vs_esn_state_ridge": mcnemar_vs_esn,
        },
        f,
        indent=2,
    )

Existing test labels match this run's test labels (same order).
Non-recurrent control vs acoustic-feature Ridge
Non-recurrent control only correct: 909
Acoustic Ridge only correct: 212
Continuity-corrected chi-square: 432.1285
Corrected p-value:             5.593e-96
Exact two-sided p-value:       3.424e-103

Non-recurrent control vs ESN-state Ridge
Non-recurrent control only correct: 244
ESN-state Ridge only correct: 365
Continuity-corrected chi-square: 23.6453
Corrected p-value:             1.158e-06
Exact two-sided p-value:       1.072e-06


## 10. RQ1 summary

Prints the three-system RQ1 comparison reported in Results Section 4.2.

In [11]:
with open(EXISTING_RUN_DIR / "results_summary_final.json") as f:
    existing_summary = json.load(f)

acoustic_test = existing_summary["rq1"]["acoustic_ridge"]["test"]
esn_test = existing_summary["rq1"]["esn_state_ridge"]["test"]

rows = [
    ("Acoustic-feature Ridge", acoustic_test),
    ("Non-recurrent control", control_test_metrics),
    ("ESN-state Ridge", esn_test),
]

print("=" * 66)
print(f"{'System':28s}{'Accuracy':>12s}{'Macro F1':>12s}{'Balanced Acc':>14s}")
print("-" * 66)
for name, m in rows:
    print(f"{name:28s}{m['acc']*100:11.2f}%{m['macro_f1']*100:11.2f}%{m['balanced_acc']*100:13.2f}%")
print("=" * 66)

System                          Accuracy    Macro F1  Balanced Acc
------------------------------------------------------------------
Acoustic-feature Ridge            47.78%      45.79%        51.84%
Non-recurrent control             64.79%      62.13%        65.86%
ESN-state Ridge                   67.75%      66.65%        70.35%
